# MIT (Mount, Install, imporT)

In [1]:
!pip install rioxarray pystac_client planetary_computer

In [2]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
#import seaborn as sns

# Data Science
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets
import xarray as xr

# Geospatial raster data handling
import rioxarray as rxr

# Geospatial data analysis
#import geopandas as gpd

# Geospatial operations
import rasterio
from rasterio import windows
from rasterio import features
from rasterio import warp
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

# Image Processing
from PIL import Image

# Coordinate transformations
from pyproj import Proj, Transformer, CRS

# Feature Engineering
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam

# Planetary Computer Tools
import pystac_client
import planetary_computer as pc
from pystac.extensions.eo import EOExtension as eo

# Others
import os
from tqdm import tqdm


In [3]:
# prompt: mount drive

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Read GT CSV (UHI Index File)

In [66]:
# prompt: read this into a df:  "/content/drive/MyDrive/uhi/Code/Data/Training_data_uhi_index_2025-02-18.csv"

import pandas as pd

uhi_df = pd.read_csv("/content/drive/MyDrive/uhi/Code/Data/Training_data_uhi_index_2025-02-18.csv")
uhi_df

,Longitude,Latitude,datetime,UHI Index
0,-73.909167,40.813107,24-07-2021 15:53,1.030289
1,-73.909187,40.813045,24-07-2021 15:53,1.030289
2,-73.909215,40.812978,24-07-2021 15:53,1.023798
3,-73.909242,40.812908,24-07-2021 15:53,1.023798
4,-73.909257,40.812845,24-07-2021 15:53,1.021634
...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245


In [67]:
# prompt: make uhi_df into gdf based on "Longitude" and "Latitude" columns

import geopandas as gpd

# Assuming 'Longitude' and 'Latitude' columns exist in uhi_df
uhi_gdf = gpd.GeoDataFrame(uhi_df, geometry=gpd.points_from_xy(uhi_df.Longitude, uhi_df.Latitude))
uhi_gdf = uhi_gdf.set_crs("EPSG:4326") #Setting the coordinate reference system to WGS84
uhi_gdf

,Longitude,Latitude,datetime,UHI Index,geometry
0,-73.909167,40.813107,24-07-2021 15:53,1.030289,POINT (-73.90917 40.81311)
1,-73.909187,40.813045,24-07-2021 15:53,1.030289,POINT (-73.90919 40.81304)
2,-73.909215,40.812978,24-07-2021 15:53,1.023798,POINT (-73.90922 40.81298)
3,-73.909242,40.812908,24-07-2021 15:53,1.023798,POINT (-73.90924 40.81291)
4,-73.909257,40.812845,24-07-2021 15:53,1.021634,POINT (-73.90926 40.81284)
...,...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470,POINT (-73.95705 40.79033)
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470,POINT (-73.95706 40.79031)
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124,POINT (-73.95709 40.79027)
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245,POINT (-73.95711 40.79025)


In [68]:
# Create a projected version for analysis
uhi_gdf_local = uhi_gdf.to_crs("EPSG:2263")  # NAD83 / New York Long Island for analysis

In [69]:
uhi_gdf_local

,Longitude,Latitude,datetime,UHI Index,geometry
0,-73.909167,40.813107,24-07-2021 15:53,1.030289,POINT (1009393.606 235526.824)
1,-73.909187,40.813045,24-07-2021 15:53,1.030289,POINT (1009388.093 235504.35)
2,-73.909215,40.812978,24-07-2021 15:53,1.023798,POINT (1009380.276 235480.052)
3,-73.909242,40.812908,24-07-2021 15:53,1.023798,POINT (1009372.92 235454.541)
4,-73.909257,40.812845,24-07-2021 15:53,1.021634,POINT (1009368.791 235431.463)
...,...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470,POINT (996143.074 227219.577)
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470,POINT (996139.388 227210.467)
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124,POINT (996131.087 227196.498)
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245,POINT (996126.012 227190.422)


# Load Buildings

In [60]:
# prompt: load Building_Footprints.csv in /content/drive/MyDrive/uhi/Code/Data

import pandas as pd

# Load the Building_Footprints.csv file
building_footprints_df = pd.read_csv('/content/drive/MyDrive/uhi/Code/Data/Building_Footprints.csv')

# Print some info to verify it loaded correctly
print(building_footprints_df.head())
building_footprints_df.info()

                                            the_geom      BIN  HEIGHTROOF  \
0  MULTIPOLYGON (((-73.96592098696333 40.76274101...  1000000      359.00   
1  MULTIPOLYGON (((-74.04443069358864 40.68965893...  1000002      119.11   
2  MULTIPOLYGON (((-74.0112906838676 40.701255197...  1000003       71.00   
3  MULTIPOLYGON (((-74.01403035921098 40.70104721...  1000004       44.19   
4  MULTIPOLYGON (((-74.01153424206372 40.70195429...  1000005      662.00   

   FEAT_CODE  GROUNDELEV  
0       1006        61.0  
1       2100        16.0  
2       2100        -1.0  
3       2100         9.0  
4       2100         6.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149563 entries, 0 to 149562
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   the_geom    149563 non-null  object 
 1   BIN         149563 non-null  int64  
 2   HEIGHTROOF  149563 non-null  float64
 3   FEAT_CODE   149563 non-null  int64  
 4   GROUNDEL

In [61]:
# prompt: building_footprints_df = pd.read_csv('/content/drive/MyDrive/uhi/Code/Data/Building_Footprints.csv')
# actually contains the geometry MULTIPOLYGON in 'the_geom' column. can we read as gdf? they are in lat/long formats

import geopandas as gpd
from shapely import wkt

# Assuming 'the_geom' column contains WKT representation of MULTIPOLYGON geometries
building_footprints_df['the_geom'] = building_footprints_df['the_geom'].apply(wkt.loads)

# Create a GeoDataFrame
building_footprints_gdf = gpd.GeoDataFrame(building_footprints_df, geometry='the_geom')

# Set the coordinate reference system to WGS84
building_footprints_gdf = building_footprints_gdf.set_crs("EPSG:4326")

# Now you can work with building_footprints_gdf as a GeoDataFrame
print(building_footprints_gdf.head())
building_footprints_gdf.info()

                                            the_geom      BIN  HEIGHTROOF  \
0  MULTIPOLYGON (((-73.96592 40.76274, -73.96601 ...  1000000      359.00   
1  MULTIPOLYGON (((-74.04443 40.68966, -74.04433 ...  1000002      119.11   
2  MULTIPOLYGON (((-74.01129 40.70126, -74.01121 ...  1000003       71.00   
3  MULTIPOLYGON (((-74.01403 40.70105, -74.01406 ...  1000004       44.19   
4  MULTIPOLYGON (((-74.01153 40.70195, -74.01152 ...  1000005      662.00   

   FEAT_CODE  GROUNDELEV  
0       1006        61.0  
1       2100        16.0  
2       2100        -1.0  
3       2100         9.0  
4       2100         6.0  
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 149563 entries, 0 to 149562
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   the_geom    149563 non-null  geometry
 1   BIN         149563 non-null  int64   
 2   HEIGHTROOF  149563 non-null  float64 
 3   FEAT_CODE   149563 non-null  int64   

In [62]:
# Create a projected version for analysis
building_footprints_gdf_local = building_footprints_gdf.to_crs("EPSG:2263")  # NAD83 / New York Long Island for analysis

In [63]:
print("Local CRS:", building_footprints_gdf_local.crs)

# Calculate the area in square feet (now that we're in a projected CRS)
building_footprints_gdf_local['building_footprint_area'] = building_footprints_gdf_local['the_geom'].area

# Calculate the centroid in the projected CRS
building_footprints_gdf_local['centroid'] = building_footprints_gdf_local['the_geom'].centroid

# Calculate building height
building_footprints_gdf_local['building_height'] = building_footprints_gdf_local['HEIGHTROOF'] - building_footprints_gdf_local['GROUNDELEV']

Local CRS: EPSG:2263


In [64]:
building_footprints_gdf_local

,the_geom,BIN,HEIGHTROOF,FEAT_CODE,GROUNDELEV,building_footprint_area,centroid,building_height
0,"MULTIPOLYGON (((993690.564 217165.691, 993665....",1000000,359.00,1006,61.0,5027.741141,POINT (993639.3 217160.147),298.00
1,"MULTIPOLYGON (((971928.31 190540.91, 971957.25...",1000002,119.11,2100,16.0,60488.729274,POINT (971897.707 190386.391),103.11
2,"MULTIPOLYGON (((981119.369 194762.846, 981141....",1000003,71.00,2100,-1.0,56371.970884,POINT (981002.014 194687.87),72.00
3,"MULTIPOLYGON (((980359.712 194687.183, 980350....",1000004,44.19,2100,9.0,8609.293820,POINT (980245.155 194695.089),35.19
4,"MULTIPOLYGON (((981051.87 195017.556, 981055.7...",1000005,662.00,2100,6.0,59966.165981,POINT (980917.837 195069.334),656.00
...,...,...,...,...,...,...,...,...
149558,"MULTIPOLYGON (((1014514.674 253183.097, 101451...",2798099,26.00,2100,56.0,3977.808160,POINT (1014536.265 253163.211),-30.00
149559,"MULTIPOLYGON (((1003622.035 240002.939, 100364...",2798105,56.00,2100,8.0,14927.332583,POINT (1003559.843 240077.168),48.00
149560,"MULTIPOLYGON (((1027717.081 245371.004, 102768...",2799101,26.00,2100,25.0,2190.651167,POINT (1027719.368 245404.596),1.00
149561,"MULTIPOLYGON (((1032053.444 250105.484, 103209...",2799360,0.00,2100,34.0,7060.018163,POINT (1032089.359 250067.654),-34.00


# Load Trees

In [30]:
# prompt: load 2015_Street_Tree_Census_-_Tree_Data_20250421.csv in /content/drive/MyDrive/uhi/Code/Data

import pandas as pd

# Load the 2015_Street_Tree_Census_-_Tree_Data_20250421.csv file
trees_df = pd.read_csv('/content/drive/MyDrive/uhi/Code/Data/2015_Street_Tree_Census_-_Tree_Data_20250421.csv')

# Print some info to verify it loaded correctly
print(trees_df.head())
trees_df.info()

   tree_id  block_id  created_at  tree_dbh  stump_diam curb_loc status health  \
0   180683    348711  08/27/2015         3           0   OnCurb  Alive   Fair   
1   200540    315986  09/03/2015        21           0   OnCurb  Alive   Fair   
2   204026    218365  09/05/2015         3           0   OnCurb  Alive   Good   
3   204337    217969  09/05/2015        10           0   OnCurb  Alive   Good   
4   189565    223043  08/30/2015        21           0   OnCurb  Alive   Good   

                            spc_latin       spc_common  ...  boro_ct  \
0                         Acer rubrum        red maple  ...  4073900   
1                   Quercus palustris          pin oak  ...  4097300   
2  Gleditsia triacanthos var. inermis      honeylocust  ...  3044900   
3  Gleditsia triacanthos var. inermis      honeylocust  ...  3044900   
4                     Tilia americana  American linden  ...  3016500   

      state   latitude  longitude         x_sp         y_sp council district  \


In [31]:
# prompt: turn trees_df into gdf with the 'latitude' and 'longitude' columns
import geopandas as gpd

# Assuming 'latitude' and 'longitude' columns exist in trees_df
trees_gdf = gpd.GeoDataFrame(
    trees_df, geometry=gpd.points_from_xy(trees_df.longitude, trees_df.latitude), crs="EPSG:4326"
)

In [32]:
trees_gdf_local = trees_gdf.to_crs("EPSG:2263")  # NAD83 / New York Long Island for analysis

In [33]:
# prompt: show all columns in df preview

pd.set_option("display.max_columns", None)

In [34]:
trees_gdf_local

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,steward,guards,sidewalk,user_type,problems,root_stone,root_grate,root_other,trunk_wire,trnk_light,trnk_other,brch_light,brch_shoe,brch_other,address,postcode,zip_city,community board,borocode,borough,cncldist,st_assem,st_senate,nta,nta_name,boro_ct,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,geometry
0,180683,348711,08/27/2015,3,0,OnCurb,Alive,Fair,Acer rubrum,red maple,NaN,NaN,NoDamage,TreesCount Staff,NaN,No,No,No,No,No,No,No,No,No,108-005 70 AVENUE,11375,Forest Hills,406,4,Queens,29,28,16,QN17,Forest Hills,4073900,New York,40.723092,-73.844215,1.027431e+06,202756.7687,29.0,739.0,4052307.0,4.022210e+09,POINT (1027431.148 202756.767)
1,200540,315986,09/03/2015,21,0,OnCurb,Alive,Fair,Quercus palustris,pin oak,NaN,NaN,Damage,TreesCount Staff,Stones,Yes,No,No,No,No,No,No,No,No,147-074 7 AVENUE,11357,Whitestone,407,4,Queens,19,27,11,QN49,Whitestone,4097300,New York,40.794111,-73.818679,1.034456e+06,228644.8374,19.0,973.0,4101931.0,4.044750e+09,POINT (1034455.701 228644.838)
2,204026,218365,09/05/2015,3,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,NaN,Damage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,390 MORGAN AVENUE,11211,Brooklyn,301,3,Brooklyn,34,50,18,BK90,East Williamsburg,3044900,New York,40.717581,-73.936608,1.001823e+06,200716.8913,34.0,449.0,3338310.0,3.028870e+09,POINT (1001822.833 200716.891)
3,204337,217969,09/05/2015,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,NaN,NaN,Damage,Volunteer,Stones,Yes,No,No,No,No,No,No,No,No,1027 GRAND STREET,11211,Brooklyn,301,3,Brooklyn,34,53,18,BK90,East Williamsburg,3044900,New York,40.713537,-73.934456,1.002420e+06,199244.2531,34.0,449.0,3338342.0,3.029250e+09,POINT (1002420.358 199244.251)
4,189565,223043,08/30/2015,21,0,OnCurb,Alive,Good,Tilia americana,American linden,NaN,NaN,Damage,Volunteer,Stones,Yes,No,No,No,No,No,No,No,No,603 6 STREET,11215,Brooklyn,306,3,Brooklyn,39,44,21,BK37,Park Slope-Gowanus,3016500,New York,40.666778,-73.975979,9.909138e+05,182202.4260,39.0,165.0,3025654.0,3.010850e+09,POINT (990913.776 182202.428)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683783,155433,217978,08/18/2015,25,0,OnCurb,Alive,Good,Quercus palustris,pin oak,NaN,NaN,Damage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,32 MARCY AVENUE,11211,Brooklyn,301,3,Brooklyn,34,53,18,BK73,North Side-South Side,3051900,New York,40.713211,-73.954944,9.967407e+05,199121.6363,34.0,519.0,3062513.0,3.023690e+09,POINT (996740.686 199121.635)
683784,183795,348185,08/29/2015,7,0,OnCurb,Alive,Good,Cladrastis kentukea,Kentucky yellowwood,1or2,NaN,NoDamage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,67-035 SELFRIDGE STREET,11375,Forest Hills,406,4,Queens,29,28,15,QN17,Forest Hills,4070700,New York,40.715194,-73.856650,1.023989e+06,199873.6475,29.0,707.0,4075448.0,4.031810e+09,POINT (1023989.074 199873.647)
683785,166161,401670,08/22/2015,12,0,OnCurb,Alive,Good,Acer rubrum,red maple,NaN,NaN,Damage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,130 BIDWELL AVENUE,10314,Staten Island,501,5,Staten Island,50,63,24,SI07,Westerleigh,5020100,New York,40.620762,-74.136517,9.463514e+05,165466.0763,50.0,201.0,5011657.0,5.004080e+09,POINT (946351.411 165466.077)
683786,184028,504204,08/29/2015,9,0,OnCurb,Alive,Good,Acer rubrum,red maple,NaN,NaN,NoDamage,TreesCount Staff,NaN,No,No,No,No,No,No,No,No,No,1985 ANTHONY AVENUE,10457,Bronx,205,2,Bronx,15,86,33,BX41,Mount Hope,2023502,New York,40.850828,-73.903115,1.011054e+06,249271.9507,15.0,23502.0,2007757.0,2.028120e+09,POINT (1011053.647 249271.952)


# Vibe (2) Current - Donut effect features: Yolo block

In [127]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import datetime

# Set buffer distances with more meaningful increments
BUFFER_DISTANCES = [25, 50, 100, 200, 500]

def find_features_in_rings(uhi_points, features_gdf, max_search_distance=500,
                          feature_type="building", size_column=None):
    """
    Find features (buildings or trees) in ring buffers around UHI points.
    Uses non-cumulative rings rather than overlapping buffers.

    Parameters:
    -----------
    uhi_points : GeoDataFrame
        GeoDataFrame containing UHI point geometries
    features_gdf : GeoDataFrame
        GeoDataFrame containing feature geometries (buildings or trees)
    max_search_distance : int
        Maximum search distance in feet
    feature_type : str
        "building" or "tree" to specify what kind of features
    size_column : str
        Column name for the size attribute (building_height or tree_dbh)
    """
    print(f"Finding {feature_type}s within {max_search_distance}ft of UHI points...")

    # Create a copy of UHI points with proper indexing
    uhi_points_reset = uhi_points.reset_index()
    uhi_points_reset.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Create a larger buffer for initial filtering (optimization)
    buffer_gdf = gpd.GeoDataFrame(
        {'uhi_id': uhi_points_reset.uhi_id},
        geometry=uhi_points_reset.geometry.buffer(max_search_distance),
        crs=uhi_points.crs
    )

    # Reset features index
    features_reset = features_gdf.reset_index(drop=True)

    # Fix potential negative values in size column
    if size_column and size_column in features_reset.columns:
        features_reset[f'safe_{size_column}'] = features_reset[size_column].clip(lower=1.0)

    # Spatial join to find candidate features
    joined = gpd.sjoin(
        features_reset,
        buffer_gdf,
        how='inner',
        predicate='intersects'
    )

    if len(joined) == 0:
        print(f"  No {feature_type}s found near any UHI points")
        return pd.DataFrame()

    print(f"  Found {len(joined)} potential {feature_type}-UHI pairs")

    # Calculate true distance from point to feature
    results = []

    for uhi_id, group in joined.groupby('uhi_id'):
        # Get the UHI point geometry
        uhi_point = uhi_points_reset[uhi_points_reset.uhi_id == uhi_id].geometry.iloc[0]

        # Calculate distance to each feature
        for idx, feature in group.iterrows():
            # Calculate true distance from point to feature
            # For buildings: point to polygon distance
            # For trees: point to point distance
            distance = uhi_point.distance(feature.geometry if feature_type == "tree" else feature.the_geom)

            # Only keep features within the maximum distance
            if distance <= max_search_distance:
                result_dict = {
                    'uhi_id': uhi_id,
                    f'{feature_type}_id': idx,
                    'distance': distance,
                }

                # Add size attributes
                if feature_type == "building":
                    result_dict.update({
                        'building_height': max(1.0, feature.building_height),
                        'building_area': feature.building_footprint_area
                    })
                elif feature_type == "tree":
                    result_dict['tree_dbh'] = feature.tree_dbh

                results.append(result_dict)

    if not results:
        print(f"  No {feature_type}s found within the maximum distance")
        return pd.DataFrame()

    # Convert to DataFrame
    result_df = pd.DataFrame(results)

    # Summary statistics
    feature_count = result_df.groupby('uhi_id').size()
    uhi_with_features = len(feature_count)

    print(f"  Found {len(result_df)} {feature_type}-UHI relationships")
    print(f"  {uhi_with_features} UHI points ({uhi_with_features/len(uhi_points)*100:.1f}%) have {feature_type}s within {max_search_distance}ft")

    return result_df

def create_ring_buffer_features(uhi_points, nearby_features_df, feature_type="building",
                              buffer_distances=BUFFER_DISTANCES):
    """
    Create features for ring buffers (non-cumulative) rather than
    overlapping cumulative buffers.
    """
    print(f"Creating {feature_type} features for ring buffers...")

    # Initialize result dataframe
    uhi_df = uhi_points.reset_index()
    uhi_df.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Create distance bins
    bins = [0] + buffer_distances
    bin_labels = [f"{bins[i]}-{bins[i+1]}ft" for i in range(len(bins)-1)]

    # Assign each feature to a distance bin
    nearby_features_df['distance_bin'] = pd.cut(
        nearby_features_df['distance'],
        bins=bins,
        labels=bin_labels,
        include_lowest=True
    )

    # Calculate count features for each ring
    for bin_label in bin_labels:
        # Count features in this ring
        features_in_ring = nearby_features_df[nearby_features_df['distance_bin'] == bin_label]
        count_by_uhi = features_in_ring.groupby('uhi_id').size()
        count_by_uhi = count_by_uhi.reset_index(name=f'{feature_type}_count_{bin_label}')

        # Calculate impact for each ring
        if feature_type == "building":
            # For buildings: (height * area) / log(distance)
            features_in_ring['contribution'] = (
                features_in_ring['building_height'] *
                features_in_ring['building_area']
            ) / np.log(np.maximum(1.1, features_in_ring['distance']))

        elif feature_type == "tree":
            # For trees: (dbh^2) / log(distance)
            features_in_ring['contribution'] = (
                features_in_ring['tree_dbh'] ** 2
            ) / np.log(np.maximum(1.1, features_in_ring['distance']))

        # Sum contributions by UHI point
        impact_by_uhi = features_in_ring.groupby('uhi_id')['contribution'].sum()
        impact_by_uhi = impact_by_uhi.reset_index(name=f'{feature_type}_impact_{bin_label}')

        # Merge count and impact with UHI dataframe
        uhi_df = uhi_df.merge(count_by_uhi, on='uhi_id', how='left')
        uhi_df = uhi_df.merge(impact_by_uhi, on='uhi_id', how='left')

        # Fill NAs with 0
        uhi_df[f'{feature_type}_count_{bin_label}'] = uhi_df[f'{feature_type}_count_{bin_label}'].fillna(0)
        uhi_df[f'{feature_type}_impact_{bin_label}'] = uhi_df[f'{feature_type}_impact_{bin_label}'].fillna(0)

    # Add closest feature
    idx = nearby_features_df.groupby('uhi_id')['distance'].idxmin()
    if len(idx) > 0:
        closest_features = nearby_features_df.loc[idx]

        if feature_type == "building":
            closest_data = closest_features[['uhi_id', 'distance', 'building_height', 'building_area']]
            closest_data.columns = [
                'uhi_id', 'closest_building_distance', 'closest_building_height', 'closest_building_area'
            ]
        elif feature_type == "tree":
            closest_data = closest_features[['uhi_id', 'distance', 'tree_dbh']]
            closest_data.columns = [
                'uhi_id', 'closest_tree_distance', 'closest_tree_dbh'
            ]

        # Merge with UHI dataframe
        uhi_df = uhi_df.merge(closest_data, on='uhi_id', how='left')

        # Fill NAs
        if feature_type == "building":
            uhi_df['closest_building_distance'] = uhi_df['closest_building_distance'].fillna(float('inf'))
            uhi_df['closest_building_height'] = uhi_df['closest_building_height'].fillna(1.0)
            uhi_df['closest_building_area'] = uhi_df['closest_building_area'].fillna(0.0)
        elif feature_type == "tree":
            uhi_df['closest_tree_distance'] = uhi_df['closest_tree_distance'].fillna(float('inf'))
            uhi_df['closest_tree_dbh'] = uhi_df['closest_tree_dbh'].fillna(0.0)

    # Print summary statistics
    print(f"\n{feature_type.capitalize()} feature summary:")
    for bin_label in bin_labels:
        count_col = f'{feature_type}_count_{bin_label}'
        impact_col = f'{feature_type}_impact_{bin_label}'

        non_zero_count = (uhi_df[count_col] > 0).sum()
        print(f"{count_col}: {non_zero_count} UHI points ({non_zero_count/len(uhi_df)*100:.1f}%)")

        non_zero_impact = (uhi_df[impact_col] > 0).sum()
        print(f"{impact_col}: {non_zero_impact} UHI points ({non_zero_impact/len(uhi_df)*100:.1f}%)")

    return uhi_df

def create_all_features_and_save(uhi_original, buildings, trees, output_dir=None):
    """
    Create all building and tree features and save to CSV with timestamp.
    """
    # Create directory if needed
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    # Find buildings near UHI points
    nearby_buildings = find_features_in_rings(
        uhi_original,
        buildings,
        max_search_distance=max(BUFFER_DISTANCES),
        feature_type="building",
        size_column="building_height"
    )

    # Find trees near UHI points
    nearby_trees = find_features_in_rings(
        uhi_original,
        trees,
        max_search_distance=max(BUFFER_DISTANCES),
        feature_type="tree",
        size_column="tree_dbh"
    )

    # Create building features
    if len(nearby_buildings) > 0:
        building_features = create_ring_buffer_features(
            uhi_original,
            nearby_buildings,
            feature_type="building",
            buffer_distances=BUFFER_DISTANCES
        )
    else:
        building_features = None

    # Create tree features
    if len(nearby_trees) > 0:
        tree_features = create_ring_buffer_features(
            uhi_original,
            nearby_trees,
            feature_type="tree",
            buffer_distances=BUFFER_DISTANCES
        )
    else:
        tree_features = None

    # Combine features
    combined = uhi_original.reset_index()
    combined.rename(columns={'index': 'uhi_id'}, inplace=True)

    if building_features is not None:
        building_cols = [col for col in building_features.columns
                        if 'building' in col and col != 'uhi_id' and col != 'geometry']
        combined = combined.merge(
            building_features[['uhi_id'] + building_cols],
            on='uhi_id',
            how='left'
        )

    if tree_features is not None:
        tree_cols = [col for col in tree_features.columns
                    if 'tree' in col and col != 'uhi_id' and col != 'geometry']
        combined = combined.merge(
            tree_features[['uhi_id'] + tree_cols],
            on='uhi_id',
            how='left'
        )

    # Fill missing values
    for col in combined.columns:
        if 'count' in col or 'impact' in col:
            combined[col] = combined[col].fillna(0)
        elif 'distance' in col:
            combined[col] = combined[col].fillna(float('inf'))
        elif 'height' in col or 'area' in col or 'dbh' in col:
            combined[col] = combined[col].fillna(0)

    # Create GeoDataFrame
    combined_gdf = gpd.GeoDataFrame(combined, geometry='geometry', crs=uhi_original.crs)

    # Convert to WGS84 if needed
    if combined_gdf.crs != "EPSG:4326":
        combined_gdf = combined_gdf.to_crs("EPSG:4326")

    # Add longitude and latitude columns
    combined_gdf['longitude'] = combined_gdf.geometry.x
    combined_gdf['latitude'] = combined_gdf.geometry.y

    # Save to file if output_dir is specified
    if output_dir:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f"{output_dir}/combined_tree_building_uhi_{timestamp}.csv"
        combined_gdf.to_csv(output_file, index=False)
        print(f"\nSaved combined dataset to: {output_file}")

    # Print summary
    print("\nFinal dataset summary:")
    print(f"Total UHI points: {len(combined_gdf)}")
    print(f"Total columns: {len(combined_gdf.columns)}")

    building_cols = [col for col in combined_gdf.columns if 'building' in col]
    tree_cols = [col for col in combined_gdf.columns if 'tree' in col]
    print(f"Building feature columns: {len(building_cols)}")
    print(f"Tree feature columns: {len(tree_cols)}")

    return combined_gdf

# Run the complete feature creation and save process
output_dir = "/content/drive/MyDrive/uhi/Code/Data/output"
combined_uhi_improved = create_all_features_and_save(
    uhi_original=uhi_gdf_local,
    buildings=building_footprints_gdf_local,
    trees=trees_gdf_local,
    output_dir=output_dir
)

Finding buildings within 500ft of UHI points...
  Found 765629 potential building-UHI pairs
  Found 765629 building-UHI relationships
  11066 UHI points (98.5%) have buildings within 500ft
Finding trees within 500ft of UHI points...
  Found 1087519 potential tree-UHI pairs
  Found 1087519 tree-UHI relationships
  10708 UHI points (95.4%) have trees within 500ft
Creating building features for ring buffers...

Building feature summary:
building_count_0-25ft: 3453 UHI points (30.8%)
building_impact_0-25ft: 3453 UHI points (30.8%)
building_count_25-50ft: 6543 UHI points (58.3%)
building_impact_25-50ft: 6543 UHI points (58.3%)
building_count_50-100ft: 9060 UHI points (80.7%)
building_impact_50-100ft: 9060 UHI points (80.7%)
building_count_100-200ft: 10334 UHI points (92.0%)
building_impact_100-200ft: 10334 UHI points (92.0%)
building_count_200-500ft: 11016 UHI points (98.1%)
building_impact_200-500ft: 11016 UHI points (98.1%)
Creating tree features for ring buffers...

Tree feature summary:

# Vibe (1) Archive -- CUMULATIVE features

## Join buildings for UHI index data

In [73]:
import numpy as np

# Step 1: Create a buffer around UHI points
# First make a copy of UHI points to preserve original data
uhi_buffer = uhi_gdf_local.copy()
uhi_buffer['buffer_geometry'] = uhi_gdf_local.geometry.buffer(30)

print("Step 1 complete: Created buffer around UHI points")
print(f"Number of UHI points: {len(uhi_buffer)}")
print(uhi_buffer.head(2))

Step 1 complete: Created buffer around UHI points
Number of UHI points: 11229
   Longitude   Latitude          datetime  UHI Index  \
0 -73.909167  40.813107  24-07-2021 15:53   1.030289   
1 -73.909187  40.813045  24-07-2021 15:53   1.030289   

                         geometry  \
0  POINT (1009393.606 235526.824)   
1   POINT (1009388.093 235504.35)   

                                     buffer_geometry  
0  POLYGON ((1009423.606 235526.824, 1009423.461 ...  
1  POLYGON ((1009418.093 235504.35, 1009417.948 2...  


In [74]:
# Step 2: Use spatial join with the buffer
# Convert buffers to a GeoDataFrame with original point geometry preserved
buffer_gdf = gpd.GeoDataFrame(
    {'uhi_id': uhi_gdf_local.index, 'uhi_point': uhi_gdf_local.geometry},
    geometry=uhi_buffer['buffer_geometry'],
    crs=uhi_gdf_local.crs
)

# Reset index to avoid duplicate index issues
buffer_gdf = buffer_gdf.reset_index(drop=True)
buildings_reset = building_footprints_gdf_local.reset_index(drop=False)

print("Step 2 complete: Prepared data for spatial join")
print(f"Buffer GeoDataFrame: {len(buffer_gdf)} rows")
print(f"Buildings GeoDataFrame: {len(buildings_reset)} rows")

Step 2 complete: Prepared data for spatial join
Buffer GeoDataFrame: 11229 rows
Buildings GeoDataFrame: 149563 rows


In [75]:
# Perform the spatial join
joined = gpd.sjoin(buildings_reset, buffer_gdf, how="inner", predicate="intersects")

print("Step 3 complete: Performed spatial join")
print(f"Number of building-buffer intersections: {len(joined)}")
print(f"Number of unique UHI points with intersections: {joined['uhi_id'].nunique()}")
print(joined.head(2))

Step 3 complete: Performed spatial join
Number of building-buffer intersections: 6481
Number of unique UHI points with intersections: 4613
       index                                           the_geom      BIN  \
13460  13460  MULTIPOLYGON (((989136.52 219279.09, 989101.05...  1026318   
13460  13460  MULTIPOLYGON (((989136.52 219279.09, 989101.05...  1026318   

       HEIGHTROOF  FEAT_CODE  GROUNDELEV  building_footprint_area  \
13460       715.0       2100        84.0            145737.925025   
13460       715.0       2100        84.0            145737.925025   

                           centroid  building_height  index_right  uhi_id  \
13460  POINT (988932.127 219250.85)            631.0         1644    1644   
13460  POINT (988932.127 219250.85)            631.0         1643    1643   

                           uhi_point  
13460  POINT (989140.828 219250.996)  
13460  POINT (989151.902 219281.967)  


In [78]:
# Step 4 (revised): Calculate distance and contribution with better handling of edge cases
# First, examine the data to see what's causing problems
print("Examining potential issues:")
print(f"Buildings with height <= 0: {(joined['building_height'] <= 0).sum()}")
print(f"Buildings with area <= 0: {(joined['building_footprint_area'] <= 0).sum()}")
print(f"Distances equal to 1 (minimum): {(joined['distance'] == 1.0).sum()}")

# Fix the contribution calculation with safeguards
joined['safe_height'] = joined['building_height'].clip(lower=0.1)  # Avoid negative or zero heights
joined['safe_area'] = joined['building_footprint_area'].clip(lower=0.1)  # Avoid negative or zero areas
joined['safe_distance'] = joined['distance'].clip(lower=1.1)  # Slightly higher min to avoid log(1)=0

# Recalculate contribution with safer values
joined['contribution'] = (
    joined['safe_height'] *
    joined['safe_area']
) / np.log(joined['safe_distance'])

# Check if we fixed the issues
print("\nRecalculated contribution statistics:")
print(f"  Min: {joined['contribution'].min()}")
print(f"  Max: {joined['contribution'].max()}")
print(f"  Mean: {joined['contribution'].mean()}")
print(f"  NaN count: {joined['contribution'].isna().sum()}")

Examining potential issues:
Buildings with height <= 0: 3185
Buildings with area <= 0: 0
Distances equal to 1 (minimum): 669

Recalculated contribution statistics:
  Min: 4.343166138851076
  Max: 114860518.9519557
  Mean: 1538674.6001336416
  NaN count: 20


In [79]:
# Step 5: Aggregate by UHI point
impact = joined.groupby('uhi_id')['contribution'].sum().reset_index()
impact.columns = ['uhi_id', 'building_impact']

print("Step 5 complete: Aggregated contributions by UHI point")
print(f"Number of UHI points with building impact: {len(impact)}")
print("Building impact statistics:")
print(f"  Min: {impact['building_impact'].min()}")
print(f"  Max: {impact['building_impact'].max()}")
print(f"  Mean: {impact['building_impact'].mean()}")


Step 5 complete: Aggregated contributions by UHI point
Number of UHI points with building impact: 4613
Building impact statistics:
  Min: 0.0
  Max: 114860518.9519557
  Mean: 2155078.385316163


In [85]:
# Alternative approach - create a new dataframe with just the essential columns
impact_lookup = dict(zip(impact['uhi_id'], impact['building_impact']))

# Create a copy of the original UHI dataframe
uhi_final_alt = uhi_gdf_local.copy()

# Map the building impact values to the UHI points based on index
# For each index, look it up in the impact_lookup dictionary, return 0 if not found
uhi_final_alt['building_impact'] = [impact_lookup.get(idx, 0) for idx in uhi_final_alt.index]

# Check the result
print("\nAlternative approach result:")
print(uhi_final_alt[['building_impact']].head(10))

# Look at some non-zero values (if any)
non_zero = uhi_final_alt[uhi_final_alt['building_impact'] > 0]
print(f"\nFound {len(non_zero)} UHI points with non-zero building impact")
if len(non_zero) > 0:
    print("Sample non-zero values:")
    print(non_zero[['building_impact']].head(5))


Alternative approach result:
   building_impact
0              0.0
1              0.0
2              0.0
3              0.0
4              0.0
5              0.0
6              0.0
7              0.0
8              0.0
9              0.0

Found 4594 UHI points with non-zero building impact
Sample non-zero values:
    building_impact
13    125009.475276
15    124462.962936
19    124576.524496
28      1895.393264
29      1898.466294


In [82]:
uhi_final

,original_index,Longitude,Latitude,datetime,UHI Index,geometry,uhi_id,building_impact
0,0,-73.909167,40.813107,24-07-2021 15:53,1.030289,POINT (1009393.606 235526.824),NaN,0.0
1,1,-73.909187,40.813045,24-07-2021 15:53,1.030289,POINT (1009388.093 235504.35),NaN,0.0
2,2,-73.909215,40.812978,24-07-2021 15:53,1.023798,POINT (1009380.276 235480.052),NaN,0.0
3,3,-73.909242,40.812908,24-07-2021 15:53,1.023798,POINT (1009372.92 235454.541),NaN,0.0
4,4,-73.909257,40.812845,24-07-2021 15:53,1.021634,POINT (1009368.791 235431.463),NaN,0.0
...,...,...,...,...,...,...,...,...
11224,11224,-73.957050,40.790333,24-07-2021 15:57,0.972470,POINT (996143.074 227219.577),NaN,0.0
11225,11225,-73.957063,40.790308,24-07-2021 15:57,0.972470,POINT (996139.388 227210.467),NaN,0.0
11226,11226,-73.957093,40.790270,24-07-2021 15:57,0.981124,POINT (996131.087 227196.498),NaN,0.0
11227,11227,-73.957112,40.790253,24-07-2021 15:59,0.981245,POINT (996126.012 227190.422),NaN,0.0


In [56]:
uhi_gdf

,Longitude,Latitude,datetime,UHI Index,geometry
0,-73.909167,40.813107,24-07-2021 15:53,1.030289,POINT (-73.90917 40.81311)
1,-73.909187,40.813045,24-07-2021 15:53,1.030289,POINT (-73.90919 40.81304)
2,-73.909215,40.812978,24-07-2021 15:53,1.023798,POINT (-73.90922 40.81298)
3,-73.909242,40.812908,24-07-2021 15:53,1.023798,POINT (-73.90924 40.81291)
4,-73.909257,40.812845,24-07-2021 15:53,1.021634,POINT (-73.90926 40.81284)
...,...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470,POINT (-73.95705 40.79033)
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470,POINT (-73.95706 40.79031)
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124,POINT (-73.95709 40.79027)
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245,POINT (-73.95711 40.79025)


In [54]:
uhi_with_feature

,index,Longitude,Latitude,datetime,UHI Index,geometry,uhi_index,building_impact_feature,uhi_point_id,building_impact
0.0,NaN,-73.909167,40.813107,24-07-2021 15:53,1.030289,POINT (1009393.606 235526.824),0,0.0,0.0,125009.475276
1.0,NaN,-73.909187,40.813045,24-07-2021 15:53,1.030289,POINT (1009388.093 235504.35),1,0.0,1.0,124462.962936
2.0,NaN,-73.909215,40.812978,24-07-2021 15:53,1.023798,POINT (1009380.276 235480.052),2,0.0,2.0,124576.524496
3.0,NaN,-73.909242,40.812908,24-07-2021 15:53,1.023798,POINT (1009372.92 235454.541),3,0.0,3.0,-48332.528221
4.0,NaN,-73.909257,40.812845,24-07-2021 15:53,1.021634,POINT (1009368.791 235431.463),4,0.0,4.0,-48410.890490
...,...,...,...,...,...,...,...,...,...,...
NaN,NaN,-73.957050,40.790333,24-07-2021 15:57,0.972470,POINT (996143.074 227219.577),11224,0.0,11224.0,0.000000
NaN,NaN,-73.957063,40.790308,24-07-2021 15:57,0.972470,POINT (996139.388 227210.467),11225,0.0,11225.0,0.000000
NaN,NaN,-73.957093,40.790270,24-07-2021 15:57,0.981124,POINT (996131.087 227196.498),11226,0.0,11226.0,0.000000
NaN,NaN,-73.957112,40.790253,24-07-2021 15:59,0.981245,POINT (996126.012 227190.422),11227,0.0,11227.0,0.000000


## NEW NEW NEW features (cleaned up)

In [107]:
import numpy as np
import pandas as pd
import geopandas as gpd

def find_buildings_near_uhi(uhi_points, buildings, max_search_distance=100):
    """Find buildings near UHI points using true distance to polygon"""
    print(f"Finding buildings within {max_search_distance}ft of UHI points...")

    # Create a copy of UHI points with proper indexing
    uhi_points_reset = uhi_points.reset_index()
    uhi_points_reset.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Create a larger buffer for initial filtering (optimization)
    buffer_gdf = gpd.GeoDataFrame(
        {'uhi_id': uhi_points_reset.uhi_id},
        geometry=uhi_points_reset.geometry.buffer(max_search_distance),
        crs=uhi_points.crs
    )

    # Reset buildings index
    buildings_reset = buildings.reset_index(drop=True)

    # Spatial join to find candidate buildings
    joined = gpd.sjoin(
        buildings_reset,
        buffer_gdf,
        how='inner',
        predicate='intersects'
    )

    if len(joined) == 0:
        print("  No buildings found near any UHI points")
        return pd.DataFrame()

    print(f"  Found {len(joined)} potential building-UHI pairs")

    # Calculate true distance from point to polygon
    results = []

    for uhi_id, group in joined.groupby('uhi_id'):
        # Get the UHI point geometry
        uhi_point = uhi_points_reset[uhi_points_reset.uhi_id == uhi_id].geometry.iloc[0]

        # Calculate distance to each building
        for idx, building in group.iterrows():
            # Calculate true distance from point to polygon
            distance = uhi_point.distance(building.the_geom)

            # Only keep buildings within the maximum distance
            if distance <= max_search_distance:
                results.append({
                    'uhi_id': uhi_id,
                    'building_id': idx,
                    'distance': distance,
                    'building_height': max(1.0, building.building_height),  # Ensure positive height
                    'building_area': building.building_footprint_area
                })

    if not results:
        print("  No buildings found within the maximum distance")
        return pd.DataFrame()

    # Convert to DataFrame
    result_df = pd.DataFrame(results)

    # Summary statistics
    building_count = result_df.groupby('uhi_id').size()
    uhi_with_buildings = len(building_count)

    print(f"  Found {len(result_df)} building-UHI relationships")
    print(f"  {uhi_with_buildings} UHI points ({uhi_with_buildings/len(uhi_points)*100:.1f}%) have buildings within {max_search_distance}ft")

    return result_df

In [108]:
def create_building_count_features(uhi_points, nearby_buildings_df, distances=[10, 20, 30, 50, 100]):
    """Create building count features for different distances"""

    # Initialize result dataframe
    uhi_df = uhi_points.reset_index()
    uhi_df.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Create count features
    for dist in distances:
        # Count buildings within this distance
        buildings_within_dist = nearby_buildings_df[nearby_buildings_df['distance'] <= dist]
        count_by_uhi = buildings_within_dist.groupby('uhi_id').size().reset_index(name=f'building_count_{dist}ft')

        # Merge with UHI dataframe
        uhi_df = uhi_df.merge(count_by_uhi, on='uhi_id', how='left')
        uhi_df[f'building_count_{dist}ft'] = uhi_df[f'building_count_{dist}ft'].fillna(0)

    return uhi_df

In [109]:
def calculate_building_impact(uhi_points, nearby_buildings_df, distances=[10, 20, 30, 50, 100]):
    """Calculate building impact feature: sum((height * area) / log(distance))"""

    # Start with dataframe from building count
    uhi_df = create_building_count_features(uhi_points, nearby_buildings_df, distances)

    # For each distance threshold, calculate building impact
    for dist in distances:
        # Filter buildings within this distance
        buildings_within_dist = nearby_buildings_df[nearby_buildings_df['distance'] <= dist]

        # Calculate impact for each building-UHI relationship
        buildings_within_dist['contribution'] = (
            buildings_within_dist['building_height'] *
            buildings_within_dist['building_area']
        ) / np.log(np.maximum(1.1, buildings_within_dist['distance']))

        # Sum contributions by UHI point
        impact_by_uhi = buildings_within_dist.groupby('uhi_id')['contribution'].sum()
        impact_by_uhi = impact_by_uhi.reset_index(name=f'building_impact_{dist}ft')

        # Merge with UHI dataframe
        uhi_df = uhi_df.merge(impact_by_uhi, on='uhi_id', how='left')
        uhi_df[f'building_impact_{dist}ft'] = uhi_df[f'building_impact_{dist}ft'].fillna(0)

    return uhi_df

In [110]:
def add_closest_building_features(uhi_df, nearby_buildings_df):
    """Add features for the closest building to each UHI point"""

    if len(nearby_buildings_df) == 0:
        return uhi_df

    # Find the closest building for each UHI point
    idx = nearby_buildings_df.groupby('uhi_id')['distance'].idxmin()
    closest_buildings = nearby_buildings_df.loc[idx]

    # Extract features
    closest_features = closest_buildings[['uhi_id', 'distance', 'building_height', 'building_area']]
    closest_features.columns = [
        'uhi_id', 'closest_building_distance', 'closest_building_height', 'closest_building_area'
    ]

    # Merge with UHI dataframe
    uhi_df = uhi_df.merge(closest_features, on='uhi_id', how='left')

    # Fill NAs with appropriate values
    uhi_df['closest_building_distance'] = uhi_df['closest_building_distance'].fillna(float('inf'))
    uhi_df['closest_building_height'] = uhi_df['closest_building_height'].fillna(1.0)
    uhi_df['closest_building_area'] = uhi_df['closest_building_area'].fillna(0.0)

    return uhi_df

In [111]:
def create_all_building_features(uhi_points, buildings, max_search_distance=100, distances=[10, 20, 30, 50, 100]):
    """Create all building features"""

    # Find buildings near UHI points
    nearby_buildings = find_buildings_near_uhi(uhi_points, buildings, max_search_distance)

    if len(nearby_buildings) == 0:
        print("No buildings found near UHI points. Returning empty features.")
        return uhi_points.copy()

    # Calculate building impact features (includes count features)
    uhi_with_features = calculate_building_impact(uhi_points, nearby_buildings, distances)

    # Add closest building features
    uhi_with_features = add_closest_building_features(uhi_with_features, nearby_buildings)

    # Display feature summary
    print("\nFeature summary:")
    for dist in distances:
        count_col = f'building_count_{dist}ft'
        impact_col = f'building_impact_{dist}ft'

        non_zero_count = (uhi_with_features[count_col] > 0).sum()
        print(f"{count_col}: {non_zero_count} UHI points ({non_zero_count/len(uhi_with_features)*100:.1f}%)")

        non_zero_impact = (uhi_with_features[impact_col] > 0).sum()
        print(f"{impact_col}: {non_zero_impact} UHI points ({non_zero_impact/len(uhi_with_features)*100:.1f}%)")

    # Drop redundant columns and reset the index
    cols_to_keep = [col for col in uhi_with_features.columns if col != 'uhi_id' or col == 'geometry']
    uhi_with_features = uhi_with_features[cols_to_keep]

    return uhi_with_features

In [112]:
# Create all building features with a maximum search distance of 100 feet
uhi_with_features = create_all_building_features(
    uhi_gdf_local,
    building_footprints_gdf_local,
    max_search_distance=100,
    distances=[10, 20, 30, 50, 100]
)

# Display sample
print("\nSample of UHI points with features:")
display_cols = [col for col in uhi_with_features.columns if 'building' in col]
# print(uhi_with_features[['uhi_id'] + display_cols].head(10))

# Save to CSV if needed
# uhi_with_features.to_csv('uhi_with_building_features.csv', index=False)

Finding buildings within 100ft of UHI points...
  Found 46734 potential building-UHI pairs
  Found 46734 building-UHI relationships
  10083 UHI points (89.8%) have buildings within 100ft

Feature summary:
building_count_10ft: 1199 UHI points (10.7%)
building_impact_10ft: 1199 UHI points (10.7%)
building_count_20ft: 2515 UHI points (22.4%)
building_impact_20ft: 2515 UHI points (22.4%)
building_count_30ft: 4614 UHI points (41.1%)
building_impact_30ft: 4614 UHI points (41.1%)
building_count_50ft: 7995 UHI points (71.2%)
building_impact_50ft: 7995 UHI points (71.2%)
building_count_100ft: 10083 UHI points (89.8%)
building_impact_100ft: 10083 UHI points (89.8%)

Sample of UHI points with features:


KeyError: "['uhi_id'] not in index"

In [115]:
uhi_with_features[display_cols].head(20)

,building_count_10ft,building_count_20ft,building_count_30ft,building_count_50ft,building_count_100ft,building_impact_10ft,building_impact_20ft,building_impact_30ft,building_impact_50ft,building_impact_100ft,closest_building_distance,closest_building_height,closest_building_area
0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.000000,50964.857372,50964.857372,42.709999,31.10,6152.544875
1,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.000000,50703.116745,50703.116745,43.545844,31.10,6152.544875
2,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.000000,50158.810946,50158.810946,45.366157,31.10,6152.544875
3,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.000000,50140.371852,430933.682359,45.429845,31.10,6152.544875
4,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.000000,50675.913604,447635.638488,43.634149,31.10,6152.544875
5,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.000000,0.000000,463324.590082,52.046370,1.00,10150.426174
6,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.000000,2670.011772,468167.705374,44.774599,1.00,10150.426174
7,0.0,0.0,0.0,1.0,5.0,0.0,0.0,0.000000,2654.662051,475909.113794,45.769720,1.00,10150.426174
8,0.0,0.0,0.0,2.0,6.0,0.0,0.0,0.000000,4619.137720,445345.925202,44.211584,1.00,7351.830375
9,0.0,0.0,0.0,2.0,7.0,0.0,0.0,0.000000,11393.459190,451863.609226,41.705023,18.57,1894.339548


## New features (ARCHIVE)

In [100]:
# # Create a simpler version focusing just on the building counts
# import numpy as np

# # Set default minimum height
# default_min_height = 1.0

# # Function to calculate just building count for a given buffer distance
# def calculate_building_count(uhi_points, buildings, buffer_distance):
#     print(f"Calculating building count for {buffer_distance}ft buffer...")

#     # Create buffer
#     uhi_buffer = uhi_points.copy()
#     uhi_buffer['buffer_geometry'] = uhi_points.geometry.buffer(buffer_distance)

#     # Prepare buffer GeoDataFrame
#     buffer_gdf = gpd.GeoDataFrame(
#         {'uhi_id': uhi_points.index, 'uhi_point': uhi_points.geometry},
#         geometry=uhi_buffer['buffer_geometry'],
#         crs=uhi_points.crs
#     )
#     buffer_gdf = buffer_gdf.reset_index(drop=True)

#     # Reset buildings index
#     buildings_reset = buildings.reset_index(drop=True)

#     # Spatial join
#     joined = gpd.sjoin(buildings_reset, buffer_gdf, how="inner", predicate="intersects")

#     if len(joined) == 0:
#         print(f"  No buildings found within {buffer_distance}ft of any UHI point")
#         return pd.DataFrame(columns=['uhi_id', f'building_count_{buffer_distance}ft'])

#     # Count buildings per UHI point
#     building_count = joined.groupby('uhi_id').size().reset_index(name=f'building_count_{buffer_distance}ft')

#     print(f"  Found {len(joined)} buildings intersecting with {len(building_count)} UHI points")
#     print(f"  Average buildings per UHI point: {building_count[f'building_count_{buffer_distance}ft'].mean():.2f}")

#     return building_count

# # Start with the UHI dataframe
# uhi_with_features = uhi_gdf_local.reset_index()
# uhi_with_features.rename(columns={'index': 'original_index'}, inplace=True)

# # Calculate building counts for different distances
# counts_30ft = calculate_building_count(uhi_gdf_local, building_footprints_gdf_local, 30)
# counts_20ft = calculate_building_count(uhi_gdf_local, building_footprints_gdf_local, 20)
# counts_10ft = calculate_building_count(uhi_gdf_local, building_footprints_gdf_local, 10)

# # Merge the counts one by one with clear suffixes
# if len(counts_30ft) > 0:
#     uhi_with_features = uhi_with_features.merge(
#         counts_30ft,
#         left_on='original_index',
#         right_on='uhi_id',
#         how='left',
#         suffixes=(None, '_30ft')
#     )
#     # Drop the duplicate uhi_id column
#     if 'uhi_id' in uhi_with_features.columns:
#         uhi_with_features.drop('uhi_id', axis=1, inplace=True)

# if len(counts_20ft) > 0:
#     uhi_with_features = uhi_with_features.merge(
#         counts_20ft,
#         left_on='original_index',
#         right_on='uhi_id',
#         how='left',
#         suffixes=(None, '_20ft')
#     )
#     # Drop the duplicate uhi_id column
#     if 'uhi_id' in uhi_with_features.columns:
#         uhi_with_features.drop('uhi_id', axis=1, inplace=True)

# if len(counts_10ft) > 0:
#     uhi_with_features = uhi_with_features.merge(
#         counts_10ft,
#         left_on='original_index',
#         right_on='uhi_id',
#         how='left',
#         suffixes=(None, '_10ft')
#     )
#     # Drop the duplicate uhi_id column
#     if 'uhi_id' in uhi_with_features.columns:
#         uhi_with_features.drop('uhi_id', axis=1, inplace=True)

# # Fill NaN values with 0
# for col in uhi_with_features.columns:
#     if 'building_count' in col:
#         uhi_with_features[col] = uhi_with_features[col].fillna(0)

# # Display summary
# print("\nBuilding count summary:")
# count_cols = [col for col in uhi_with_features.columns if 'count' in col]
# for col in count_cols:
#     non_zero = (uhi_with_features[col] > 0).sum()
#     print(f"{col}: {non_zero} UHI points with buildings ({non_zero/len(uhi_with_features)*100:.1f}%)")

# # Show a sample of UHI points with buildings
# has_buildings = uhi_with_features[uhi_with_features['building_count_30ft'] > 0]
# print(f"\nSample of {min(5, len(has_buildings))} UHI points with buildings nearby:")
# if len(has_buildings) > 0:
#     print(has_buildings[['original_index'] + count_cols].head())

# # Show a sample of UHI points without buildings
# no_buildings = uhi_with_features[uhi_with_features['building_count_30ft'] == 0]
# print(f"\nSample of {min(5, len(no_buildings))} UHI points without buildings nearby:")
# if len(no_buildings) > 0:
#     print(no_buildings[['original_index'] + count_cols].head())

# print(f"\nFinal dataset shape: {uhi_with_features.shape}")

Calculating building count for 30ft buffer...
  Found 6481 buildings intersecting with 4613 UHI points
  Average buildings per UHI point: 1.40
Calculating building count for 20ft buffer...
  Found 3245 buildings intersecting with 2511 UHI points
  Average buildings per UHI point: 1.29
Calculating building count for 10ft buffer...
  Found 1400 buildings intersecting with 1199 UHI points
  Average buildings per UHI point: 1.17

Building count summary:
building_count_30ft: 4613 UHI points with buildings (41.1%)
building_count_20ft: 2511 UHI points with buildings (22.4%)
building_count_10ft: 1199 UHI points with buildings (10.7%)

Sample of 5 UHI points with buildings nearby:
    original_index  building_count_30ft  building_count_20ft  \
13              13                  1.0                  0.0   
15              15                  1.0                  0.0   
19              19                  1.0                  0.0   
28              28                  1.0                  0.0   

Calculating building impact for 30ft buffer...
  Calculated impact for 4613 UHI points
  Min impact: 0.00
  Max impact: 114860518.95
  Mean impact: 2161744.80
Calculating building impact for 20ft buffer...
  Calculated impact for 2511 UHI points
  Min impact: 0.00
  Max impact: 114860518.95
  Mean impact: 3739484.25
Calculating building impact for 10ft buffer...
  Calculated impact for 1199 UHI points
  Min impact: 0.00
  Max impact: 114860518.95
  Mean impact: 7544123.88

Building impact summary:
building_impact_30ft: 4594 UHI points with non-zero value (40.9%)
  Min (non-zero): 47.19
  Max: 114860518.95
  Mean (overall): 888069.17
building_impact_20ft: 2502 UHI points with non-zero value (22.3%)
  Min (non-zero): 47.19
  Max: 114860518.95
  Mean (overall): 836213.82
building_impact_10ft: 1196 UHI points with non-zero value (10.7%)
  Min (non-zero): 194.78
  Max: 114860518.95
  Mean (overall): 805539.63

Sample of 5 UHI points with building impact:
    original_index  building_count_3

In [103]:
# def find_closest_building(uhi_points, buildings, max_search_distance=100, min_height=1.0):
#     print(f"Finding closest building (search radius: {max_search_distance}ft)...")

#     # Create a copy of the UHI points with index column
#     uhi_points_reset = uhi_points.copy().reset_index()
#     uhi_points_reset.rename(columns={'index': 'uhi_id'}, inplace=True)

#     # Create a spatial index for buildings to speed up operations
#     print("  Creating spatial index for buildings...")
#     buildings_reset = buildings.reset_index(drop=True)
#     buildings_reset['safe_height'] = buildings_reset['building_height'].clip(lower=min_height)

#     # Initialize an empty dataframe for results
#     result_df = pd.DataFrame(columns=['uhi_id', 'closest_building_distance',
#                                       'closest_building_height', 'closest_building_area'])

#     # Process in batches to avoid memory issues
#     batch_size = 1000
#     total_points = len(uhi_points_reset)

#     for start_idx in range(0, total_points, batch_size):
#         end_idx = min(start_idx + batch_size, total_points)
#         batch = uhi_points_reset.iloc[start_idx:end_idx]

#         print(f"  Processing batch {start_idx//batch_size + 1}/{(total_points + batch_size - 1)//batch_size}...")

#         # For each UHI point, find the closest building
#         batch_results = []

#         for _, uhi_row in batch.iterrows():
#             uhi_point = uhi_row.geometry
#             uhi_id = uhi_row.uhi_id

#             # Find buildings within buffer
#             buffer = uhi_point.buffer(max_search_distance)
#             nearby_buildings = buildings_reset[buildings_reset.the_geom.intersects(buffer)]

#             if len(nearby_buildings) == 0:
#                 continue

#             # Calculate distances
#             distances = [uhi_point.distance(building.the_geom) for _, building in nearby_buildings.iterrows()]
#             min_dist_idx = np.argmin(distances)
#             closest_building = nearby_buildings.iloc[min_dist_idx]

#             batch_results.append({
#                 'uhi_id': uhi_id,
#                 'closest_building_distance': distances[min_dist_idx],
#                 'closest_building_height': closest_building.safe_height,
#                 'closest_building_area': closest_building.building_footprint_area
#             })

#         # Add batch results to main results
#         if batch_results:
#             batch_df = pd.DataFrame(batch_results)
#             result_df = pd.concat([result_df, batch_df], ignore_index=True)

#     print(f"  Found closest buildings for {len(result_df)} UHI points")
#     if len(result_df) > 0:
#         print(f"  Average distance to closest building: {result_df['closest_building_distance'].mean():.2f}ft")

#     return result_df

# closest_building = find_closest_building(uhi_gdf_local, building_footprints_gdf_local)

Finding closest building (search radius: 100ft)...
  Creating spatial index for buildings...
  Processing batch 1/12...
  Processing batch 2/12...
  Processing batch 3/12...
  Processing batch 4/12...
  Processing batch 5/12...
  Processing batch 6/12...
  Processing batch 7/12...
  Processing batch 8/12...
  Processing batch 9/12...
  Processing batch 10/12...
  Processing batch 11/12...
  Processing batch 12/12...
  Found closest buildings for 10083 UHI points
  Average distance to closest building: 34.63ft


In [95]:
# # Combine all features into a single dataframe
# print("\nMerging all features...")

# # Start with the UHI dataframe
# uhi_with_features = uhi_gdf_local.reset_index()
# uhi_with_features.rename(columns={'index': 'original_index'}, inplace=True)

# # Merge in all the features
# feature_dataframes = [
#     features_30ft,
#     features_20ft,
#     features_10ft,
#     closest_building
# ]

# for feature_df in feature_dataframes:
#     if len(feature_df) > 0:
#         uhi_with_features = uhi_with_features.merge(
#             feature_df,
#             left_on='original_index',
#             right_on='uhi_id',
#             how='left'
#         )

# # Fill NaN values
# columns_to_fill_zero = [
#     'building_impact_30ft',
#     'building_count_30ft',
#     'building_impact_20ft',
#     'building_count_20ft',
#     'building_impact_10ft',
#     'building_count_10ft',
#     'closest_building_area'
# ]

# columns_to_fill_default = {
#     'closest_building_distance': float('inf'),
#     'closest_building_height': default_min_height
# }

# for col in columns_to_fill_zero:
#     if col in uhi_with_features.columns:
#         uhi_with_features[col] = uhi_with_features[col].fillna(0)

# for col, default_val in columns_to_fill_default.items():
#     if col in uhi_with_features.columns:
#         uhi_with_features[col] = uhi_with_features[col].fillna(default_val)

# # Display summary
# print("\nFeatures summary:")
# count_columns = [col for col in uhi_with_features.columns if 'count' in col]
# impact_columns = [col for col in uhi_with_features.columns if 'impact' in col]
# closest_columns = [col for col in uhi_with_features.columns if 'closest' in col]

# for col in count_columns:
#     if col in uhi_with_features.columns:
#         non_zero = (uhi_with_features[col] > 0).sum()
#         print(f"{col}: {non_zero} UHI points with buildings ({non_zero/len(uhi_with_features)*100:.1f}%)")

# for col in impact_columns:
#     if col in uhi_with_features.columns:
#         non_zero = (uhi_with_features[col] > 0).sum()
#         print(f"{col}: {non_zero} UHI points with non-zero value ({non_zero/len(uhi_with_features)*100:.1f}%)")

# # Show sample of points with buildings nearby
# has_buildings = uhi_with_features[uhi_with_features['building_count_30ft'] > 0]
# print(f"\nSample of {min(5, len(has_buildings))} UHI points with buildings nearby:")
# if len(has_buildings) > 0:
#     display_cols = ['original_index'] + count_columns + impact_columns + closest_columns
#     display_cols = [col for col in display_cols if col in uhi_with_features.columns]
#     print(has_buildings[display_cols].head())

# # Show sample of points without buildings nearby
# no_buildings = uhi_with_features[uhi_with_features['building_count_30ft'] == 0]
# print(f"\nSample of {min(5, len(no_buildings))} UHI points without buildings nearby:")
# if len(no_buildings) > 0:
#     print(no_buildings[display_cols].head())

# # Clean up columns
# columns_to_keep = ['original_index', 'geometry'] + [col for col in uhi_with_features.columns if col not in ['uhi_id', 'uhi_point', 'index']]
# uhi_with_features = uhi_with_features[columns_to_keep]

# print(f"\nFinal dataset shape: {uhi_with_features.shape}")
# print(f"Columns: {uhi_with_features.columns.tolist()}")


Merging all features...


MergeError: Passing 'suffixes' which cause duplicate columns {'uhi_id_x'} is not allowed.

In [92]:
# uhi_with_features['building_impact_20ft'].unique()

array([    0.        , 22187.20483478,   342.95541175, ...,
       15751.51721001,  3317.78457182,  3197.02509977])

## New New features (better algo - ARCHIVE)

In [104]:
# def find_nearest_buildings(uhi_points, buildings, max_search_distance=100, n_nearest=3):
#     print(f"Finding {n_nearest} nearest buildings (max distance: {max_search_distance}ft)...")

#     # Create a larger buffer for initial filtering
#     uhi_buffer = uhi_points.copy()
#     uhi_buffer['buffer_geometry'] = uhi_points.geometry.buffer(max_search_distance)

#     # Reset indices
#     uhi_points_reset = uhi_points.reset_index()
#     uhi_points_reset.rename(columns={'index': 'uhi_id'}, inplace=True)
#     buildings_reset = buildings.reset_index()

#     # Create a spatial index on buildings for faster querying
#     buildings_reset = buildings_reset.copy()

#     # Prepare result dataframe
#     results = []

#     # Process in batches to prevent memory issues
#     batch_size = 1000
#     n_batches = (len(uhi_points_reset) + batch_size - 1) // batch_size

#     for i in range(n_batches):
#         start_idx = i * batch_size
#         end_idx = min((i + 1) * batch_size, len(uhi_points_reset))
#         batch = uhi_points_reset.iloc[start_idx:end_idx]

#         print(f"  Processing batch {i+1}/{n_batches}...")

#         # Get buffer for this batch
#         buffer_batch = gpd.GeoDataFrame(
#             geometry=batch.geometry.buffer(max_search_distance),
#             data={'uhi_id': batch.uhi_id},
#             crs=uhi_points.crs
#         )

#         # Find candidate buildings using spatial join
#         joined = gpd.sjoin(
#             buildings_reset,
#             buffer_batch,
#             how='inner',
#             predicate='intersects'
#         )

#         if len(joined) == 0:
#             continue

#         # Calculate exact distances from UHI points to buildings
#         batch_results = []
#         for uhi_id in batch.uhi_id:
#             # Get the UHI point
#             uhi_point = batch[batch.uhi_id == uhi_id].geometry.iloc[0]

#             # Find buildings that might be near this UHI point
#             candidates = joined[joined.uhi_id == uhi_id]

#             if len(candidates) == 0:
#                 continue

#             # Calculate actual distance from point to polygon
#             candidates['distance'] = candidates.apply(
#                 lambda row: uhi_point.distance(row['the_geom']),
#                 axis=1
#             )

#             # Sort by distance and take n_nearest
#             nearest = candidates.sort_values('distance').head(n_nearest)

#             # Add to results
#             for _, b in nearest.iterrows():
#                 batch_results.append({
#                     'uhi_id': uhi_id,
#                     'building_id': b.index,
#                     'building_distance': b.distance,
#                     'building_height': b.building_height,
#                     'building_area': b.building_footprint_area
#                 })

#         results.extend(batch_results)

#     # Convert to dataframe
#     if not results:
#         return pd.DataFrame(columns=['uhi_id', 'building_id', 'building_distance',
#                                      'building_height', 'building_area'])

#     results_df = pd.DataFrame(results)

#     # Create summary statistics
#     n_uhi_with_buildings = results_df['uhi_id'].nunique()
#     print(f"Found {len(results_df)} building-UHI relationships for {n_uhi_with_buildings} UHI points")
#     print(f"Coverage: {n_uhi_with_buildings/len(uhi_points)*100:.1f}% of UHI points have at least one building within {max_search_distance}ft")

#     return results_df

In [105]:
# def calculate_building_features(nearest_buildings_df, uhi_points):
#     """Calculate building features from nearest buildings dataframe"""

#     # Group by UHI point
#     grouped = nearest_buildings_df.groupby('uhi_id')

#     # Calculate features
#     features = {}

#     # 1. Number of buildings within different distances
#     for distance in [10, 20, 30, 50, 100]:
#         within_distance = nearest_buildings_df[nearest_buildings_df['building_distance'] <= distance]
#         count_by_uhi = within_distance.groupby('uhi_id').size()

#         # Convert to dictionary
#         count_dict = count_by_uhi.to_dict()

#         # Add to features
#         features[f'building_count_{distance}ft'] = count_dict

#     # 2. Weighted building impact
#     # For each UHI point, calculate sum((height * area) / log(distance))
#     def calc_impact(group):
#         return sum(
#             (row.building_height * row.building_area) /
#             np.log(max(1.1, row.building_distance))
#             for _, row in group.iterrows()
#         )

#     impact_by_uhi = grouped.apply(calc_impact)
#     features['building_impact'] = impact_by_uhi.to_dict()

#     # 3. Closest building features
#     closest_buildings = nearest_buildings_df.loc[
#         nearest_buildings_df.groupby('uhi_id')['building_distance'].idxmin()
#     ]

#     features['closest_building_distance'] = dict(
#         zip(closest_buildings.uhi_id, closest_buildings.building_distance)
#     )

#     features['closest_building_height'] = dict(
#         zip(closest_buildings.uhi_id, closest_buildings.building_height)
#     )

#     features['closest_building_area'] = dict(
#         zip(closest_buildings.uhi_id, closest_buildings.building_area)
#     )

#     # Create a dataframe with all UHI points
#     uhi_df = uhi_points.reset_index().rename(columns={'index': 'uhi_id'})

#     # Add features
#     for feature_name, feature_dict in features.items():
#         uhi_df[feature_name] = uhi_df['uhi_id'].map(feature_dict).fillna(0)

#         # For distance features, replace 0 with infinity
#         if 'distance' in feature_name and feature_name != 'building_count_distance':
#             uhi_df[feature_name] = uhi_df[feature_name].replace(0, float('inf'))

#     return uhi_df

In [106]:
# # Find nearest buildings with a larger search radius
# nearest_buildings = find_nearest_buildings(
#     uhi_gdf_local,
#     building_footprints_gdf_local,
#     max_search_distance=150,  # Increased radius
#     n_nearest=5               # Keep 5 nearest buildings per UHI point
# )

# # Calculate features
# uhi_with_features = calculate_building_features(nearest_buildings, uhi_gdf_local)

# # Display summary
# print("\nFeatures summary:")
# for col in uhi_with_features.columns:
#     if 'building' in col:
#         if 'count' in col:
#             non_zero = (uhi_with_features[col] > 0).sum()
#             print(f"{col}: {non_zero} UHI points ({non_zero/len(uhi_with_features)*100:.1f}%)")
#         elif 'distance' in col:
#             finite = (uhi_with_features[col] < float('inf')).sum()
#             if finite > 0:
#                 print(f"{col}: {finite} UHI points ({finite/len(uhi_with_features)*100:.1f}%), " +
#                      f"mean: {uhi_with_features[uhi_with_features[col] < float('inf')][col].mean():.2f}ft")
#         else:
#             non_zero = (uhi_with_features[col] > 0).sum()
#             if non_zero > 0:
#                 print(f"{col}: {non_zero} UHI points ({non_zero/len(uhi_with_features)*100:.1f}%), " +
#                      f"mean: {uhi_with_features[uhi_with_features[col] > 0][col].mean():.2f}")

Finding 5 nearest buildings (max distance: 150ft)...
  Processing batch 1/12...
  Processing batch 2/12...
  Processing batch 3/12...
  Processing batch 4/12...
  Processing batch 5/12...
  Processing batch 6/12...
  Processing batch 7/12...
  Processing batch 8/12...
  Processing batch 9/12...
  Processing batch 10/12...
  Processing batch 11/12...
  Processing batch 12/12...
Found 45022 building-UHI relationships for 10493 UHI points
Coverage: 93.4% of UHI points have at least one building within 150ft

Features summary:
building_count_10ft: 1199 UHI points (10.7%)
building_count_20ft: 2515 UHI points (22.4%)
building_count_30ft: 4614 UHI points (41.1%)
building_count_50ft: 7995 UHI points (71.2%)
building_count_100ft: 10086 UHI points (89.8%)
building_impact: 6683 UHI points (59.5%), mean: 2192487.70
closest_building_distance: 9884 UHI points (88.0%), mean: 40.27ft
closest_building_height: 6010 UHI points (53.5%), mean: 67.88
closest_building_area: 10493 UHI points (93.4%), mean: 17

## Join trees for UHI index data

In [116]:
trees_gdf

,tree_id,block_id,created_at,tree_dbh,stump_diam,curb_loc,status,health,spc_latin,spc_common,steward,guards,sidewalk,user_type,problems,root_stone,root_grate,root_other,trunk_wire,trnk_light,trnk_other,brch_light,brch_shoe,brch_other,address,postcode,zip_city,community board,borocode,borough,cncldist,st_assem,st_senate,nta,nta_name,boro_ct,state,latitude,longitude,x_sp,y_sp,council district,census tract,bin,bbl,geometry
0,180683,348711,08/27/2015,3,0,OnCurb,Alive,Fair,Acer rubrum,red maple,NaN,NaN,NoDamage,TreesCount Staff,NaN,No,No,No,No,No,No,No,No,No,108-005 70 AVENUE,11375,Forest Hills,406,4,Queens,29,28,16,QN17,Forest Hills,4073900,New York,40.723092,-73.844215,1.027431e+06,202756.7687,29.0,739.0,4052307.0,4.022210e+09,POINT (-73.84422 40.72309)
1,200540,315986,09/03/2015,21,0,OnCurb,Alive,Fair,Quercus palustris,pin oak,NaN,NaN,Damage,TreesCount Staff,Stones,Yes,No,No,No,No,No,No,No,No,147-074 7 AVENUE,11357,Whitestone,407,4,Queens,19,27,11,QN49,Whitestone,4097300,New York,40.794111,-73.818679,1.034456e+06,228644.8374,19.0,973.0,4101931.0,4.044750e+09,POINT (-73.81868 40.79411)
2,204026,218365,09/05/2015,3,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,1or2,NaN,Damage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,390 MORGAN AVENUE,11211,Brooklyn,301,3,Brooklyn,34,50,18,BK90,East Williamsburg,3044900,New York,40.717581,-73.936608,1.001823e+06,200716.8913,34.0,449.0,3338310.0,3.028870e+09,POINT (-73.93661 40.71758)
3,204337,217969,09/05/2015,10,0,OnCurb,Alive,Good,Gleditsia triacanthos var. inermis,honeylocust,NaN,NaN,Damage,Volunteer,Stones,Yes,No,No,No,No,No,No,No,No,1027 GRAND STREET,11211,Brooklyn,301,3,Brooklyn,34,53,18,BK90,East Williamsburg,3044900,New York,40.713537,-73.934456,1.002420e+06,199244.2531,34.0,449.0,3338342.0,3.029250e+09,POINT (-73.93446 40.71354)
4,189565,223043,08/30/2015,21,0,OnCurb,Alive,Good,Tilia americana,American linden,NaN,NaN,Damage,Volunteer,Stones,Yes,No,No,No,No,No,No,No,No,603 6 STREET,11215,Brooklyn,306,3,Brooklyn,39,44,21,BK37,Park Slope-Gowanus,3016500,New York,40.666778,-73.975979,9.909138e+05,182202.4260,39.0,165.0,3025654.0,3.010850e+09,POINT (-73.97598 40.66678)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683783,155433,217978,08/18/2015,25,0,OnCurb,Alive,Good,Quercus palustris,pin oak,NaN,NaN,Damage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,32 MARCY AVENUE,11211,Brooklyn,301,3,Brooklyn,34,53,18,BK73,North Side-South Side,3051900,New York,40.713211,-73.954944,9.967407e+05,199121.6363,34.0,519.0,3062513.0,3.023690e+09,POINT (-73.95494 40.71321)
683784,183795,348185,08/29/2015,7,0,OnCurb,Alive,Good,Cladrastis kentukea,Kentucky yellowwood,1or2,NaN,NoDamage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,67-035 SELFRIDGE STREET,11375,Forest Hills,406,4,Queens,29,28,15,QN17,Forest Hills,4070700,New York,40.715194,-73.856650,1.023989e+06,199873.6475,29.0,707.0,4075448.0,4.031810e+09,POINT (-73.85665 40.71519)
683785,166161,401670,08/22/2015,12,0,OnCurb,Alive,Good,Acer rubrum,red maple,NaN,NaN,Damage,Volunteer,NaN,No,No,No,No,No,No,No,No,No,130 BIDWELL AVENUE,10314,Staten Island,501,5,Staten Island,50,63,24,SI07,Westerleigh,5020100,New York,40.620762,-74.136517,9.463514e+05,165466.0763,50.0,201.0,5011657.0,5.004080e+09,POINT (-74.13652 40.62076)
683786,184028,504204,08/29/2015,9,0,OnCurb,Alive,Good,Acer rubrum,red maple,NaN,NaN,NoDamage,TreesCount Staff,NaN,No,No,No,No,No,No,No,No,No,1985 ANTHONY AVENUE,10457,Bronx,205,2,Bronx,15,86,33,BX41,Mount Hope,2023502,New York,40.850828,-73.903115,1.011054e+06,249271.9507,15.0,23502.0,2007757.0,2.028120e+09,POINT (-73.90311 40.85083)


In [117]:
def find_trees_near_uhi(uhi_points, trees, max_search_distance=100):
    """Find trees near UHI points using true distance"""
    print(f"Finding trees within {max_search_distance}ft of UHI points...")

    # Create a copy of UHI points with proper indexing
    uhi_points_reset = uhi_points.reset_index()
    uhi_points_reset.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Create a buffer for initial filtering (optimization)
    buffer_gdf = gpd.GeoDataFrame(
        {'uhi_id': uhi_points_reset.uhi_id},
        geometry=uhi_points_reset.geometry.buffer(max_search_distance),
        crs=uhi_points.crs
    )

    # Reset trees index
    trees_reset = trees.reset_index(drop=True)

    # Spatial join to find candidate trees
    joined = gpd.sjoin(
        trees_reset,
        buffer_gdf,
        how='inner',
        predicate='intersects'
    )

    if len(joined) == 0:
        print("  No trees found near any UHI points")
        return pd.DataFrame()

    print(f"  Found {len(joined)} potential tree-UHI pairs")

    # Calculate true distance from point to tree
    results = []

    for uhi_id, group in joined.groupby('uhi_id'):
        # Get the UHI point geometry
        uhi_point = uhi_points_reset[uhi_points_reset.uhi_id == uhi_id].geometry.iloc[0]

        # Calculate distance to each tree
        for idx, tree in group.iterrows():
            # Calculate true distance from UHI point to tree point
            distance = uhi_point.distance(tree.geometry)

            # Only keep trees within the maximum distance
            if distance <= max_search_distance:
                results.append({
                    'uhi_id': uhi_id,
                    'tree_id': idx,
                    'distance': distance,
                    'tree_dbh': tree.tree_dbh  # Tree diameter at breast height
                })

    if not results:
        print("  No trees found within the maximum distance")
        return pd.DataFrame()

    # Convert to DataFrame
    result_df = pd.DataFrame(results)

    # Summary statistics
    tree_count = result_df.groupby('uhi_id').size()
    uhi_with_trees = len(tree_count)

    print(f"  Found {len(result_df)} tree-UHI relationships")
    print(f"  {uhi_with_trees} UHI points ({uhi_with_trees/len(uhi_points)*100:.1f}%) have trees within {max_search_distance}ft")

    return result_df

In [118]:
def create_tree_count_features(uhi_points, nearby_trees_df, distances=[10, 20, 30, 50, 100]):
    """Create tree count features for different distances"""

    # Initialize result dataframe
    uhi_df = uhi_points.reset_index()
    uhi_df.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Create count features
    for dist in distances:
        # Count trees within this distance
        trees_within_dist = nearby_trees_df[nearby_trees_df['distance'] <= dist]
        count_by_uhi = trees_within_dist.groupby('uhi_id').size().reset_index(name=f'tree_count_{dist}ft')

        # Merge with UHI dataframe
        uhi_df = uhi_df.merge(count_by_uhi, on='uhi_id', how='left')
        uhi_df[f'tree_count_{dist}ft'] = uhi_df[f'tree_count_{dist}ft'].fillna(0)

    return uhi_df

In [119]:
def calculate_tree_impact(uhi_points, nearby_trees_df, distances=[10, 20, 30, 50, 100]):
    """Calculate tree impact feature: sum((tree_dbh^2) / log(distance))"""

    # Start with dataframe from tree count
    uhi_df = create_tree_count_features(uhi_points, nearby_trees_df, distances)

    # For each distance threshold, calculate tree impact
    for dist in distances:
        # Filter trees within this distance
        trees_within_dist = nearby_trees_df[nearby_trees_df['distance'] <= dist]

        # Calculate impact for each tree-UHI relationship
        # Square the DBH to approximate canopy area
        trees_within_dist['contribution'] = (
            trees_within_dist['tree_dbh'] ** 2
        ) / np.log(np.maximum(1.1, trees_within_dist['distance']))

        # Sum contributions by UHI point
        impact_by_uhi = trees_within_dist.groupby('uhi_id')['contribution'].sum()
        impact_by_uhi = impact_by_uhi.reset_index(name=f'tree_impact_{dist}ft')

        # Merge with UHI dataframe
        uhi_df = uhi_df.merge(impact_by_uhi, on='uhi_id', how='left')
        uhi_df[f'tree_impact_{dist}ft'] = uhi_df[f'tree_impact_{dist}ft'].fillna(0)

    return uhi_df

In [120]:
def add_closest_tree_features(uhi_df, nearby_trees_df):
    """Add features for the closest tree to each UHI point"""

    if len(nearby_trees_df) == 0:
        return uhi_df

    # Find the closest tree for each UHI point
    idx = nearby_trees_df.groupby('uhi_id')['distance'].idxmin()
    closest_trees = nearby_trees_df.loc[idx]

    # Extract features
    closest_features = closest_trees[['uhi_id', 'distance', 'tree_dbh']]
    closest_features.columns = [
        'uhi_id', 'closest_tree_distance', 'closest_tree_dbh'
    ]

    # Merge with UHI dataframe
    uhi_df = uhi_df.merge(closest_features, on='uhi_id', how='left')

    # Fill NAs with appropriate values
    uhi_df['closest_tree_distance'] = uhi_df['closest_tree_distance'].fillna(float('inf'))
    uhi_df['closest_tree_dbh'] = uhi_df['closest_tree_dbh'].fillna(0.0)

    return uhi_df

In [121]:
def create_all_tree_features(uhi_points, trees, max_search_distance=100, distances=[10, 20, 30, 50, 100]):
    """Create all tree features"""

    # Find trees near UHI points
    nearby_trees = find_trees_near_uhi(uhi_points, trees, max_search_distance)

    if len(nearby_trees) == 0:
        print("No trees found near UHI points. Returning empty features.")
        return uhi_points.copy()

    # Calculate tree impact features (includes count features)
    uhi_with_features = calculate_tree_impact(uhi_points, nearby_trees, distances)

    # Add closest tree features
    uhi_with_features = add_closest_tree_features(uhi_with_features, nearby_trees)

    # Display feature summary
    print("\nFeature summary:")
    for dist in distances:
        count_col = f'tree_count_{dist}ft'
        impact_col = f'tree_impact_{dist}ft'

        non_zero_count = (uhi_with_features[count_col] > 0).sum()
        print(f"{count_col}: {non_zero_count} UHI points ({non_zero_count/len(uhi_with_features)*100:.1f}%)")

        non_zero_impact = (uhi_with_features[impact_col] > 0).sum()
        print(f"{impact_col}: {non_zero_impact} UHI points ({non_zero_impact/len(uhi_with_features)*100:.1f}%)")

    # Clean up columns
    cols_to_keep = [col for col in uhi_with_features.columns if col != 'uhi_id' or col == 'geometry']
    uhi_with_features = uhi_with_features[cols_to_keep]

    return uhi_with_features

In [122]:
# Create all tree features with a maximum search distance of 100 feet
uhi_with_tree_features = create_all_tree_features(
    uhi_gdf_local,
    trees_gdf_local,
    max_search_distance=100,
    distances=[10, 20, 30, 50, 100]
)

# Display sample
print("\nSample of UHI points with tree features:")
display_cols = [col for col in uhi_with_tree_features.columns if 'tree' in col]
print(uhi_with_tree_features[display_cols].head(10))

Finding trees within 100ft of UHI points...
  Found 63191 potential tree-UHI pairs
  Found 63191 tree-UHI relationships
  9528 UHI points (84.9%) have trees within 100ft

Feature summary:
tree_count_10ft: 821 UHI points (7.3%)
tree_impact_10ft: 814 UHI points (7.2%)
tree_count_20ft: 3003 UHI points (26.7%)
tree_impact_20ft: 2946 UHI points (26.2%)
tree_count_30ft: 5121 UHI points (45.6%)
tree_impact_30ft: 5056 UHI points (45.0%)
tree_count_50ft: 7616 UHI points (67.8%)
tree_impact_50ft: 7560 UHI points (67.3%)
tree_count_100ft: 9528 UHI points (84.9%)
tree_impact_100ft: 9500 UHI points (84.6%)

Sample of UHI points with tree features:
   tree_count_10ft  tree_count_20ft  tree_count_30ft  tree_count_50ft  \
0              0.0              0.0              0.0              1.0   
1              0.0              0.0              0.0              0.0   
2              0.0              0.0              0.0              1.0   
3              0.0              1.0              1.0             

## Merge and Export

In [123]:
def combine_features_and_save(uhi_original, building_features, tree_features, output_file='uhi_with_all_features.csv'):
    """
    Combine building and tree features with the original UHI GeoDataFrame,
    project back to WGS84 (world coordinates), and save as CSV.

    Parameters:
    -----------
    uhi_original : GeoDataFrame
        The original UHI GeoDataFrame in WGS84 coordinates
    building_features : GeoDataFrame
        GeoDataFrame with building features
    tree_features : GeoDataFrame
        GeoDataFrame with tree features
    output_file : str
        Path to save the output CSV file
    """
    print("Combining features and preparing final dataset...")

    # Reset index on original UHI data to ensure it has uhi_id column
    uhi_with_id = uhi_original.reset_index()
    uhi_with_id.rename(columns={'index': 'uhi_id'}, inplace=True)

    # Extract just the feature columns from building_features
    building_cols = [col for col in building_features.columns
                    if 'building' in col and col != 'uhi_id']

    # Extract just the feature columns from tree_features
    tree_cols = [col for col in tree_features.columns
                if 'tree' in col and col != 'uhi_id']

    # Get the geometry from the original UHI data
    combined = uhi_with_id.copy()

    # Merge building features
    if len(building_cols) > 0:
        building_features_reset = building_features.reset_index()
        if 'uhi_id' not in building_features_reset.columns:
            building_features_reset.rename(columns={'index': 'uhi_id'}, inplace=True)

        combined = combined.merge(
            building_features_reset[['uhi_id'] + building_cols],
            on='uhi_id',
            how='left'
        )

        # Fill NA values for building features
        for col in building_cols:
            if 'count' in col or 'impact' in col:
                combined[col] = combined[col].fillna(0)
            elif 'distance' in col:
                combined[col] = combined[col].fillna(float('inf'))
            else:
                combined[col] = combined[col].fillna(0)

        print(f"Added {len(building_cols)} building feature columns")

    # Merge tree features
    if len(tree_cols) > 0:
        tree_features_reset = tree_features.reset_index()
        if 'uhi_id' not in tree_features_reset.columns:
            tree_features_reset.rename(columns={'index': 'uhi_id'}, inplace=True)

        combined = combined.merge(
            tree_features_reset[['uhi_id'] + tree_cols],
            on='uhi_id',
            how='left'
        )

        # Fill NA values for tree features
        for col in tree_cols:
            if 'count' in col or 'impact' in col:
                combined[col] = combined[col].fillna(0)
            elif 'distance' in col:
                combined[col] = combined[col].fillna(float('inf'))
            else:
                combined[col] = combined[col].fillna(0)

        print(f"Added {len(tree_cols)} tree feature columns")

    # Ensure the GeoDataFrame has the correct CRS (should be the same as original)
    combined = gpd.GeoDataFrame(combined, geometry='geometry', crs=uhi_original.crs)

    # If current CRS is not WGS84, convert it
    if combined.crs != "EPSG:4326":
        print(f"Projecting from {combined.crs} to WGS84 (EPSG:4326)...")
        combined = combined.to_crs("EPSG:4326")

    # Save to CSV
    # Note: saving to CSV loses the geometry information
    # For full GIS data, save as GeoJSON, Shapefile, or other GIS format
    combined.to_csv(output_file, index=False)
    print(f"Saved combined dataset with {len(combined)} rows and {len(combined.columns)} columns to {output_file}")

    # If you want the geometry to be preserved in a text-friendly way
    # Add longitude and latitude columns
    combined['longitude'] = combined.geometry.x
    combined['latitude'] = combined.geometry.y

    # Return the combined dataframe
    return combined

# Usage
combined_uhi = combine_features_and_save(
    uhi_original=uhi_gdf,  # Original UHI GeoDataFrame in WGS84
    building_features=uhi_with_features,  # Building features in projected coords
    tree_features=uhi_with_tree_features,  # Tree features in projected coords
    output_file='uhi_with_building_tree_features.csv'
)

# Display summary of the combined data
print("\nFinal dataset summary:")
print(f"Total UHI points: {len(combined_uhi)}")

# Count columns by type
building_cols = [col for col in combined_uhi.columns if 'building' in col]
tree_cols = [col for col in combined_uhi.columns if 'tree' in col]
print(f"Building feature columns: {len(building_cols)}")
print(f"Tree feature columns: {len(tree_cols)}")

# Display sample
print("\nSample of combined dataset (first 5 rows):")
sample_cols = ['uhi_id', 'longitude', 'latitude']
# Add a few building and tree columns
if building_cols:
    sample_cols.extend(building_cols[:2])
if tree_cols:
    sample_cols.extend(tree_cols[:2])
print(combined_uhi[sample_cols].head())

Combining features and preparing final dataset...
Added 13 building feature columns
Added 12 tree feature columns
Saved combined dataset with 11229 rows and 31 columns to uhi_with_building_tree_features.csv

Final dataset summary:
Total UHI points: 11229
Building feature columns: 13
Tree feature columns: 12

Sample of combined dataset (first 5 rows):
   uhi_id  longitude   latitude  building_count_10ft  building_count_20ft  \
0       0 -73.909167  40.813107                  0.0                  0.0   
1       1 -73.909187  40.813045                  0.0                  0.0   
2       2 -73.909215  40.812978                  0.0                  0.0   
3       3 -73.909242  40.812908                  0.0                  0.0   
4       4 -73.909257  40.812845                  0.0                  0.0   

   tree_count_10ft  tree_count_20ft  
0              0.0              0.0  
1              0.0              0.0  
2              0.0              0.0  
3              0.0              

In [125]:
import os
import datetime

# Create output directory if it doesn't exist
output_dir = "/content/drive/MyDrive/uhi/Code/Data/output"
os.makedirs(output_dir, exist_ok=True)

# Generate timestamp for filename
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"{output_dir}/combined_tree_building_uhi_{timestamp}.csv"

# Use the combine function with the timestamp in the filename
combined_uhi = combine_features_and_save(
    uhi_original=uhi_gdf,  # Original UHI GeoDataFrame in WGS84
    building_features=uhi_with_features,  # Building features
    tree_features=uhi_with_tree_features,  # Tree features
    output_file=output_filename
)

print(f"\nFile saved with timestamp: {output_filename}")

Combining features and preparing final dataset...
Added 13 building feature columns
Added 12 tree feature columns
Saved combined dataset with 11229 rows and 31 columns to /content/drive/MyDrive/uhi/Code/Data/output/combined_tree_building_uhi_20250422_012559.csv

File saved with timestamp: /content/drive/MyDrive/uhi/Code/Data/output/combined_tree_building_uhi_20250422_012559.csv
